# 🧪 BioCirv AI Analysis Playground

Welcome to the BioCirv AI analysis environment. This notebook allows you to explore the biocirv project data using natural language queries powered by PandasAI and the CBORG LLM gateway.

## 🚀 Getting Started

### 1. Initialize Environment
Run the cell below to set up the connection to GCP and initialize the AI agent. This cell handles dependency installation, repository cloning, and authentication.

In [ ]:
# 🚀 BioCirv AI - Nuclear Setup (Google Colab)

import os
import sys

def is_colab():
    return 'google.colab' in sys.modules

# 1. Environment Verification
if is_colab():
    if sys.version_info >= (3, 12):
        print("❌ ERROR: Google Colab is running Python 3.12+.")
        print("PandasAI 3.0 dependencies (scipy 1.10.1) are currently incompatible with Python 3.12.")
        print("Please switch to a Python 3.11 runtime (Runtime -> Change runtime type).")
        sys.exit(1)

    # Check if we've already done the nuclear install in this session
    if not os.path.exists('/tmp/.biocirv_initialized'):
        print("☢️ Performing Nuclear Reset of scientific stack to ensure binary compatibility...")
        
        # Uninstall EVERYTHING that might conflict
        !pip uninstall -y pandasai pandas-ai pandas numpy scipy matplotlib pillow packaging -q
        
        # Force install the exact 'Legacy track' required by PandasAI 3.0
        !pip install --force-reinstall "numpy<2.0" "scipy>=1.10.1,<1.11" "matplotlib>=3.7.1,<3.8" "pillow>=10.1.0,<11.0.0" "packaging<25" -q
        
        # Mark as initialized
        with open('/tmp/.biocirv_initialized', 'w') as f: f.write('1')
        
        print("\n♻️ Restarting Runtime to load legacy binary stack... (The session will crash, this is normal)")
        import os
        os.kill(os.getpid(), 9)

# --- If we reach here, we are in a clean Python 3.11 session with correct binaries ---

os.environ['INSTANCE_CONNECTION_NAME'] = 'biocirv-470318:us-west1:biocirv-staging'
os.environ['DB_IAM_USER'] = 'biocirv-staging-cr-worker@biocirv-470318.iam'
os.environ['DB_NAME'] = 'biocirv-staging'
os.environ['CLOUD_MODE'] = 'true'

print("🌐 Cloning repository...")
repo_path = '/content/biocirv-ai'
if os.path.exists(repo_path):
    !rm -rf {repo_path}
!git clone -b dev https://github.com/petercarbsmith/biocirv-ai.git -q

print("📦 Installing project from manifest...")
# Use --pre to allow pandasai 3.0.0 beta/stable installs
!pip install --pre -e {repo_path} pg8000 cloud-sql-python-connector -q

print("✅ Environment Synchronized!")

# 2. Configure Python Path
src_path = os.path.join(repo_path, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# 3. Run Initialization Modules
from ca_biositing.ai_exploration.colab_setup import setup_colab
setup_colab()

from ca_biositing.ai_exploration.sandbox_setup import init_sandbox, get_agent

# 4. Initialize Sandbox & Agent
llm, db_config = init_sandbox(cloud_mode=True)
agent = get_agent(llm, db_config)

print("\n✅ BioCirv AI Agent Ready!")

### 🛠️ Pre-flight Check
If you encounter issues, run this cell to verify the environment solve and module accessibility.

In [ ]:
# 1. Verify Dependencies
print("🔍 Verifying AI Stack...")
try:
    import pandasai
    import pandasai_sql
    from pandasai_sql import PostgreSQLConnector
    import pg8000
    from google.cloud.sql.connector import Connector
    print(f"✅ PandasAI: {pandasai.__version__}")
    print(f"✅ SQL Connector: Found")
    print(f"✅ GCP Connector: Found")
except ImportError as e:
    print(f"❌ Missing Module: {e}")
    print("Please re-run the initialization cell above.")

# 2. Verify Repo Path
import ca_biositing
print(f"✅ ca_biositing module loaded from: {ca_biositing.__file__}")

## 🔍 Starter Queries

Try running some of these queries to see the 'Trinity' output (Code, Data, Plot).

In [ ]:
# Query 1: Data Summary
result = agent.chat("Show me a summary of the available views in the ca_biositing schema.")
result.display()

In [ ]:
# Query 2: Visualization
result = agent.chat("Create a bar chart of the top 10 counties by biomass potential.")
result.display()

In [ ]:
# Query 3: Complex Analysis
result = agent.chat("Which counties have both high biomass potential and are within 50 miles of a major highway? Show the top 5.")
result.display()

## 🛠️ Advanced Usage

You can inspect the generated SQL and Python code for any query by looking at the `code` attribute of the result.